# Chapter 1 — Classical Systems and Information

Programming companion to Dr. Ashutosh Bhatia's transcript-led lecture notes.

We use the same route as the chapter: a familiar system, a representation, a small calculation and its meaning. **Predict → calculate by hand → run → compare → explain.** An assertion is a check on a selected calculation; it does not replace the written proofs.

This chapter uses Python, NumPy and Matplotlib for classical models. Qiskit enters when we reach quantum circuits. No hardware account is needed.

Sources: foundations slides, PDF pp.3–13; 26 July 2026 lecture (systems, sealed envelope, four bit functions, asymmetric operation and composition); August conditioning discussions. The complete joint table and additional calculations are newly authored supporting examples. All distributions are columns; operation matrices are column-stochastic. Joint tables have a separate convention: rows are X and columns are Y.

In Colab, run the setup cell first. Then work through the cells in order. Changing an example input may also require revisiting the predictions and the numerical checks below it.


In [ ]:
%pip -q install "numpy>=1.26,<3" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("NumPy:", np.__version__)
rng = np.random.default_rng(545)

def is_distribution(p):
    p = np.asarray(p, dtype=float)
    return (p.ndim == 1 and p.size > 0 and np.all(np.isfinite(p))
            and np.all(p >= 0) and np.isclose(p.sum(), 1))

def is_column_stochastic(M):
    M = np.asarray(M, dtype=float)
    return (M.ndim == 2 and M.size > 0 and np.all(np.isfinite(M))
            and np.all(M >= 0) and np.allclose(M.sum(axis=0), 1))


## 1.1 What information describes the switch?

Use the ordered labels **off, low, medium, high**. Before running the next cell, write the column for medium and its fixed binary code. Now reverse the coordinate order. Does reordering operate on the device, or change the fixed code?

The column is a coordinate representation. The binary code is a separate assignment of strings to labels.


In [ ]:
labels = ["off", "low", "medium", "high"]
selected = "medium"
code_map = {"off": "00", "low": "01", "medium": "10", "high": "11"}
basis = np.eye(len(labels), dtype=int)
v = basis[:, labels.index(selected)]
reverse = list(reversed(labels))
v_reverse = basis[:, reverse.index(selected)]
print("Coordinate order:", labels, "column:", v)
print("Reversed order:", reverse, "column:", v_reverse)
print("Fixed binary code:", code_map[selected])
assert np.array_equal(v, [0, 0, 1, 0])
assert np.array_equal(v_reverse, [0, 1, 0, 0])
assert code_map[selected] == "10"
assert len(v) == 4 and len(code_map[selected]) == 2


**An added extension.** With only off, low and high, two bits still suffice, but one string is unused. Predict the answer for five labels, then compare coordinate dimension and minimum fixed-length storage.


In [ ]:
for count in [1, 3, 4, 5, 6]:
    bits = (count - 1).bit_length()
    unused = 2**bits - count
    print(f"{count} labels: {count} one-hot coordinates; {bits} bits; {unused} unused strings")
assert (5 - 1).bit_length() == 3
# A die face label need not equal its array index.
die_labels = [1, 2, 3, 4, 5, 6]
print("Face 3 is at array index", die_labels.index(3))
assert die_labels.index(3) == 2


## 1.2 What do we know before opening the envelope?

Declare **pass ↦ 0**, **fail ↦ 1**. The column $(3/4,1/4)^T$ describes our uncertainty about the recorded result. Predict the information update after observing fail.

This example concerns one fixed letter. It is classical uncertainty, not a coherent quantum superposition, and updating our description does not require changing the printed result.


In [ ]:
p = np.array([3/4, 1/4])
result_labels = ["pass", "fail"]
observed = "fail"
updated_information = np.eye(2)[:, result_labels.index(observed)]
print("Before opening:", p)
print("Observed:", observed)
print("Updated information:", updated_information)
assert is_distribution(p)
assert np.array_equal(updated_information, [0, 1])
assert not is_distribution([1.2, -0.2])
assert not is_distribution([0.2, 0.3])


### Fresh preparations and finite counts

Now change the experiment: independently prepare a new bit on every trial, with fixed P(1)=1/4. Rereading one letter would not create these fresh trials.

Predict the expected count in 100 trials. A sampled batch may differ. Repeat the cell to see another batch; restart with the same random seed to reproduce the sequence.

The lecture's changing-team example asks a separate question: would the same fixed model remain appropriate if the preparation conditions changed?


In [ ]:
N = 100
ones = int(rng.binomial(N, p[1]))
counts = np.array([N - ones, ones])
print("Model:", p, "expected counts:", N * p)
print("Observed counts:", counts, "frequencies:", counts / N)
assert counts.sum() == N
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].bar(["0", "1"], p, color="#286a9b")
axes[0].set(title="Model probabilities", ylabel="Probability", ylim=(0, 1))
axes[1].bar(["0", "1"], counts, color="#286a9b")
axes[1].set(title=f"One sampled batch (N={N})", ylabel="Count", ylim=(0, N))
for ax in axes:
    ax.set_xlabel("Outcome")
fig.tight_layout()
plt.show()


### Events and conditional probabilities

For the weighted die in the notes, first calculate P(even), then the six-component conditional distribution. Which entries survive? What must they add up to?


In [ ]:
die = np.array([1, 2, 1, 3, 2, 3], dtype=float) / 12
even = np.array([False, True, False, True, False, True])
prob_even = die[even].sum()
conditional = np.where(even, die, 0) / prob_even
print("P(even):", prob_even, "conditional:", conditional)
assert is_distribution(die) and is_distribution(conditional)
assert np.isclose(prob_even, 2/3)
assert np.allclose(conditional, [0, 1/4, 0, 3/8, 0, 3/8])


### Learn X and update the description of Y

The joint table has rows X=0,1 and columns Y=0,1:

| X | Y=0 | Y=1 |
|---|---|---|
| 0 | 1/12 | 1/4 |
| 1 | 1/3 | 1/3 |

All four entries sum to one. **This is not an operation matrix:** its columns are not input-conditioned transition distributions.

Calculate Y's marginal distribution. Then learn X=0, retain that row, and normalize. Repeat for X=1. Compare with independent fair bits.


In [ ]:
def condition_on_row(joint, observed_x):
    joint = np.asarray(joint, dtype=float)
    if joint.ndim != 2 or joint.size == 0 or not is_distribution(joint.ravel()):
        raise ValueError("Use a normalized nonnegative joint table.")
    if not isinstance(observed_x, (int, np.integer)) or not 0 <= observed_x < len(joint):
        raise ValueError("Choose a valid observed X label.")
    event_probability = joint[observed_x].sum()
    if event_probability == 0:
        raise ValueError("The selected event has probability zero; conditioning is undefined.")
    return event_probability, joint[observed_x] / event_probability

joint = np.array([[1/12, 1/4], [1/3, 1/3]])
marginal_y = joint.sum(axis=0)
event_probability, conditional_y = condition_on_row(joint, 0)
print("Marginal P(Y):", marginal_y)
print("P(X=0):", event_probability, "P(Y | X=0):", conditional_y)
assert np.allclose(marginal_y, [5/12, 7/12])
assert np.isclose(event_probability, 1/3)
assert np.allclose(conditional_y, [1/4, 3/4])
assert np.allclose(condition_on_row(joint, 1)[1], [1/2, 1/2])
independent = np.full((2, 2), 1/4)
assert np.allclose(condition_on_row(independent, 0)[1], independent.sum(axis=0))


In [ ]:
# A zero-probability event does not have a conditional distribution here.
try:
    condition_on_row([[0, 0], [1/2, 1/2]], 0)
except ValueError as error:
    print("Boundary:", error)
else:
    raise AssertionError("Conditioning on an impossible event must be rejected.")


**Explain:** why do we divide surviving classical probabilities by P(X=0)? Later quantum conditional-state calculations will normalize amplitudes using a square root. Do not apply that later amplitude rule to this probability table.


## 1.3 Build the operation from its definite inputs

List all four deterministic bit functions first. Now use the lecture's random rule: **0 stays 0; input 1 becomes 0 or 1 equally**.

Predict each output column and build A. Explain why choosing identity or reset-to-zero with equal probabilities produces A, while applying them in sequence produces reset.


In [ ]:
I = np.eye(2)
X = np.array([[0., 1.], [1., 0.]])
R0 = np.array([[1., 1.], [0., 0.]])
R1 = np.array([[0., 0.], [1., 1.]])
A = np.array([[1., 1/2], [0., 1/2]])
e0, e1 = I[:, 0], I[:, 1]
print("A on definite input 0:", A @ e0)
print("A on definite input 1:", A @ e1)
assert np.allclose(A @ e0, [1, 0])
assert np.allclose(A @ e1, [1/2, 1/2])
assert np.allclose(A, (I + R0) / 2)
assert np.allclose(R0 @ I, R0)
assert not np.allclose(A, R0 @ I)


### An uncertain input is a weighted combination of routes

Apply A to p=(3/4,1/4). Write the two contributions to output 0 before running the code. Then compare with a symmetric random flip of probability 1/4, keeping the input fixed.


In [ ]:
p = np.array([3/4, 1/4])
epsilon = 1/4
B = (1 - epsilon) * I + epsilon * X
print("A:", A, "output Ap:", A @ p)
print("Symmetric random flip B:", B, "output Bp:", B @ p)
assert np.allclose(A @ p, [7/8, 1/8])
assert np.allclose(B @ p, [5/8, 3/8])
for M in [I, X, R0, R1, A, B]:
    assert is_column_stochastic(M)
    assert is_distribution(M @ p)
B_inverse = np.linalg.inv(B)
assert np.allclose(B_inverse @ B, I)
assert not is_column_stochastic(B_inverse)
assert np.linalg.matrix_rank(R0) == 1
print("Algebraic inverse of B (not an allowed stochastic reversal):", B_inverse)


**Try it:** change the input-1 output probabilities to 1/3 for 0 and 2/3 for 1, while keeping input 0 unchanged. Apply that operation to (1/4,3/4). Show why the answer is (1/2,1/2). Write it in a new cell so the original worked examples remain reproducible.


## 1.4 Connect two devices

Start with an intermediate state, not a memorized product order. Predict reset then flip, and flip then reset. A then B acts as B @ A on a column.

The next cell also returns to the asymmetric rule: apply A then X, and compare with X then A.


In [ ]:
reset_then_flip = X @ R0
flip_then_reset = R0 @ X
print("After reset:", R0 @ p, "then flip:", X @ (R0 @ p))
print("After flip:", X @ p, "then reset:", R0 @ (X @ p))
assert np.allclose(reset_then_flip, R1)
assert np.allclose(flip_then_reset, R0)
assert not np.allclose(reset_then_flip, flip_then_reset)
print("After A:", A @ p, "then X:", X @ (A @ p))
print("After X:", X @ p, "then A:", A @ (X @ p))
assert np.allclose(X @ (A @ p), [1/8, 7/8])
assert np.allclose(A @ (X @ p), [5/8, 3/8])
assert np.allclose(X @ (A @ p), (X @ A) @ p)
assert np.allclose(X @ B, B @ X)
uniform = np.array([1/2, 1/2])
assert np.allclose(I @ uniform, X @ uniform)
assert not np.allclose(I, X)


In [ ]:
# Selected numerical checks complement the general closure proof in the notes.
operations = [I, X, R0, R1, A, B]
for first in operations:
    for second in operations:
        product = second @ first
        assert is_column_stochastic(product)
        assert np.allclose(second @ (first @ p), product @ p)
print("All 36 operation-pair checks passed.")


## Explain the chapter without the code

1. Distinguish a state label, a basis coordinate and a storage code.
2. Explain what changed when we opened the envelope, and what did not need to change.
3. Give an assumption behind a probability estimated from historical data.
4. Explain the joint-table normalization step, including the impossible-event boundary.
5. Interpret each column of A and its output on the uncertain preparation.
6. Distinguish a random choice of devices from connecting them in sequence.
7. Explain why agreement on one input does not prove two operations equal.

We will return to the four one-bit functions in Deutsch's problem. For now they establish a precise classical foundation. This notebook makes no quantum-advantage claim.
